# battle_ship キャラ立ち絵ジェネレーター（Google Colab 無料 GPU）

Hugging Face のアニメ系 SDXL モデル（既定: [Animagine XL 4.0](https://huggingface.co/cagliostrolab/animagine-xl-4.0)）で艦カードの立ち絵を生成し、背景を切り抜いて zip で持ち帰ります。

1. メニュー **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選ぶ
2. 手元で `cd backend && go run ./cmd/portraits prompts` を実行し、`.cache/portraits/prompts.json` を作る
3. 上から順にセルを実行（③ で `prompts.json` をアップロード）
4. ⑤ の候補を見て ⑥ の `PICKS` で採用する絵を選び、⑥ を実行すると `portraits.zip` がダウンロードされる
5. 手元で `cd backend && go run ./cmd/portraits import ~/Downloads/portraits.zip`

T4 では 1 枚およそ 40 秒です。無料枠の GPU 時間には上限があるので、候補数（`CANDIDATES`）は控えめに。途中で切断されても続きから再開できるよう、`SAVE_TO_DRIVE` で Google ドライブに保存できます。

In [ ]:
#@title ① セットアップ
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U diffusers transformers accelerate "rembg[cpu]"

In [ ]:
#@title ② 設定
MODEL_ID = "cagliostrolab/animagine-xl-4.0"  #@param {type:"string"}
CANDIDATES = 3  #@param {type:"integer"}
STEPS = 28  #@param {type:"integer"}
CFG = 5.0  #@param {type:"number"}
WIDTH, HEIGHT = 832, 1216  # SDXL が得意な縦長解像度
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/battle_ship_portraits"
else:
    WORK = "/content/portraits"
RAW, OUT = f"{WORK}/candidates", f"{WORK}/cutout"
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("作業フォルダ:", WORK)

In [ ]:
#@title ③ prompts.json をアップロード
import json
from google.colab import files

uploaded = files.upload()
jobs = json.loads(next(iter(uploaded.values())))
JOBS = {j["id"]: j for j in jobs}
print(f"{len(jobs)} 枚分:", ", ".join(JOBS))

In [ ]:
#@title ④ モデル読み込み（初回は数分かかります）
import torch
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

# lpw_stable_diffusion_xl は CLIP の 77 トークン制限を超える長いプロンプトを分割して扱える
pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    custom_pipeline="lpw_stable_diffusion_xl",
    add_watermarker=False,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)
print("ready:", MODEL_ID)

In [ ]:
#@title ⑤ 候補を生成
#@markdown 生成済みの候補はスキップします。気に入らないカードは `ONLY` に ID を入れ、`SEED_OFFSET` を変えて再実行すると引き直せます。
ONLY = ""  #@param {type:"string"}
SEED_OFFSET = 0  #@param {type:"integer"}

import time
from PIL import Image, ImageDraw
from IPython.display import display

def contact_sheet(card_id, scale=0.25):
    ims = [Image.open(f"{RAW}/{card_id}_{i}.png") for i in range(CANDIDATES) if os.path.exists(f"{RAW}/{card_id}_{i}.png")]
    w, h = int(WIDTH * scale), int(HEIGHT * scale)
    sheet = Image.new("RGB", (w * len(ims), h + 24), "white")
    d = ImageDraw.Draw(sheet)
    for i, im in enumerate(ims):
        sheet.paste(im.convert("RGB").resize((w, h)), (i * w, 24))
        d.text((i * w + 6, 6), f"{card_id}={i}", fill="black")
    return sheet

only = [s.strip() for s in ONLY.split(",") if s.strip()]
for card_id in only:
    assert card_id in JOBS, f"prompts.json に {card_id} がありません"
for job in jobs:
    if only and job["id"] not in only:
        continue
    for i in range(CANDIDATES):
        path = f"{RAW}/{job['id']}_{i}.png"
        if os.path.exists(path) and not only:
            continue
        t = time.time()
        g = torch.Generator("cuda").manual_seed(job["seed"] + SEED_OFFSET + i)
        img = pipe(
            prompt=job["prompt"], negative_prompt=job["negative"],
            width=WIDTH, height=HEIGHT, num_inference_steps=STEPS, guidance_scale=CFG,
            generator=g,
        ).images[0]
        img.save(path)
        print(f"{job['id']}_{i}  {time.time() - t:.0f}s")
    print(job["id"], job["name"])
    display(contact_sheet(job["id"]))

In [ ]:
#@title ⑥ 採用する絵を選んで切り抜き → portraits.zip をダウンロード
#@markdown `PICKS` は `カードID=候補番号` をカンマ区切りで（例: `bb_guren=2, ca_soyo=1`）。指定しないカードは候補 0 を使います。
PICKS = ""  #@param {type:"string"}
SCALE = 0.75  #@param {type:"number"}

import shutil
from rembg import new_session, remove

picks = {}
for item in filter(None, (s.strip() for s in PICKS.split(","))):
    card_id, n = item.split("=")
    picks[card_id.strip()] = int(n)

# 背景が複雑だと人物を見つけられず薄い影だけ残ることがある。不透明画素がこれ未満なら失敗扱い
MIN_SOLID = 0.08
session = new_session("isnet-anime")
failed = []
for card_id in JOBS:
    src = f"{RAW}/{card_id}_{picks.get(card_id, 0)}.png"
    if not os.path.exists(src):
        continue
    img = Image.open(src)
    try:
        # alpha_matting で髪の毛先を硬く切らずに柔らかく残す
        cut = remove(img, session=session, alpha_matting=True)
    except Exception:
        cut = remove(img, session=session)
    alpha = cut.getchannel("A").histogram()
    if sum(alpha[221:]) / (cut.width * cut.height) < MIN_SOLID:
        failed.append(card_id)
        continue
    cut = cut.resize((int(cut.width * SCALE), int(cut.height * SCALE)), Image.LANCZOS)
    cut.save(f"{OUT}/{card_id}.png")
    print("cut out", card_id)

if failed:
    print("切り抜きに失敗（別の候補を PICKS で選ぶか、⑤ で引き直してください）:", ", ".join(failed))

# 暗い背景に並べてプレビュー
done = [f for f in sorted(os.listdir(OUT)) if f.endswith(".png")]
if done:
    th = 240
    tiles = [Image.open(f"{OUT}/{f}") for f in done]
    tiles = [t.resize((int(t.width * th / t.height), th)) for t in tiles]
    per_row = 8
    tw = max(t.width for t in tiles)
    preview = Image.new("RGBA", (tw * min(per_row, len(tiles)), th * ((len(tiles) - 1) // per_row + 1)), (20, 32, 54, 255))
    for k, t in enumerate(tiles):
        preview.alpha_composite(t, ((k % per_row) * tw, (k // per_row) * th))
    display(preview)

shutil.make_archive("/content/portraits", "zip", OUT)
files.download("/content/portraits.zip")